<!--nav--> [🗺 Learning path](README.md) · **25/40** · ◀ [Quantized Serving Showdown](./Quantized_Serving_Showdown.ipynb) · [Serving Internals Visualized](./Serving_Internals_Visualized_D3.ipynb) ▶

# Speculative Decoding & the Serving Frontier

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sugeerth/gpu-training-notebooks/blob/main/Speculative_Decoding_Advanced_Serving.ipynb)

The fundamentals notebook left a loose end: during decode, a GPU serving one user runs at **~1–2%
compute utilization** — the memory bus is saturated, the ALUs are bored. Speculative decoding is the
beautiful answer: **use the idle compute to guess several future tokens, then verify all guesses in
a single forward pass.** Guessed right → you got N tokens for the price of one weight-read. Guessed
wrong → you throw the wrong ones away and you're exactly where plain decoding would be.

The part people don't believe until they see the math: with the right accept/reject rule, the output
distribution is **provably identical** to the target model's — this is a *lossless* speedup, not an
approximation.

| Part | What happens |
|---|---|
| **1** | The algorithm + the acceptance math (with an interactive plot of expected speedup) |
| **2** | **Measure it**: Qwen2.5-3B target + Qwen2.5-0.5B draft, HF assisted generation, live on a T4 |
| **3** | **Prompt-lookup (n-gram) decoding** — a free draft model hiding inside your prompt |
| **4** | The 2025 state of the art: EAGLE-3, Medusa, MTP — and spec decode in vLLM |
| **5** | The serving frontier: chunked prefill, prefill/decode disaggregation, KV offload, RadixAttention |

**Runs on:** free Colab **T4** (Part 1 is pure Python, runs anywhere).

## Part 1 · The algorithm

Two models: a big **target** (the one whose quality you're paying for) and a small **draft**
(~10–100× cheaper). Each round:

```
1. DRAFT:  small model autoregressively proposes k tokens        (k cheap sequential passes)
           prompt → "The cat sat on the"  →  [ mat, and, purred ]

2. VERIFY: target model runs ONE forward pass over all k at once  (parallel — like prefill!)
           and produces its own probability for each position     (1 expensive pass)

3. ACCEPT: left to right, accept draft token x with prob min(1, p_target(x)/p_draft(x));
           at the first rejection, resample that position from the residual distribution
           max(0, p_target − p_draft) (normalized) and discard the rest.

           [ mat ✓, and ✓, purred ✗→"then" ]   →  emitted "mat and then":  3 tokens, 1 target pass
```

Step 3's rule is a small rejection-sampling proof: the emitted tokens are distributed *exactly* as
if the target model had sampled them itself. The draft can be terrible — you lose speed, never
quality. Even on full rejection you emit 1 token (the resample), so a round never produces nothing.

**How fast is it?** If the target accepts each draft token independently with probability α, the
expected tokens emitted per round with k drafts is a truncated geometric series:

$$E[\text{tokens/round}] = \frac{1 - \alpha^{k+1}}{1 - \alpha}$$

Wall-clock speedup divides that by the round's cost, `k·c + 1`, where `c` = draft cost relative to
one target pass. Two design tensions live in that formula — let's look at them:

In [ ]:
# Pure Python + matplotlib - runs anywhere, no GPU needed.
import matplotlib.pyplot as plt
import numpy as np

def expected_tokens(alpha, k):          # E[tokens emitted per verify round]
    return (1 - alpha ** (k + 1)) / (1 - alpha) if alpha < 1 else k + 1

def speedup(alpha, k, c=0.05):          # c = draft pass cost / target pass cost (~0.5B vs 3B here)
    return expected_tokens(alpha, k) / (k * c + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.2))
alphas = np.linspace(0, 0.99, 100)
for k in (1, 3, 5, 8):
    ax1.plot(alphas, [expected_tokens(a, k) for a in alphas], label=f"k={k}")
    ax2.plot(alphas, [speedup(a, k) for a in alphas], label=f"k={k}")
ax1.set(xlabel="acceptance rate α", ylabel="E[tokens per target pass]",
        title="Tokens per round (the win)")
ax2.set(xlabel="acceptance rate α", ylabel="wall-clock speedup",
        title="Speedup incl. draft cost (c=0.05)")
for ax in (ax1, ax2): ax.legend(); ax.grid(alpha=0.3)
ax2.axhline(1, color="gray", ls="--", lw=1)
plt.tight_layout(); plt.show()

print("Reading the right plot:")
print(" - α ≈ 0.6-0.8 (typical draft) with k=3-5  →  ~2-3x for free")
print(" - low α: bigger k HURTS (you pay drafts that get rejected) - below the gray line means slower!")
print(" - this is why engines auto-tune k and why acceptance rate is THE metric to watch")

## Part 2 · Measure it: 3B target, 0.5B draft, one T4

`transformers` calls this **assisted generation**. The draft must share the target's tokenizer —
model families are made for this (Qwen2.5 0.5B↔3B here; think Llama-3.2-1B drafting for
Llama-3.3-70B in production).

We test three prompt styles on purpose. Acceptance rate — and therefore speedup — depends on how
*predictable* the target's output is to a small model: code and formulaic prose speculate
beautifully; creative sampling at high temperature doesn't.

In [ ]:
!pip install -q -U transformers accelerate

import torch, time
from transformers import AutoModelForCausalLM, AutoTokenizer
assert torch.cuda.is_available(), "GPU required - Runtime > Change runtime type > T4"

TARGET, DRAFT = "Qwen/Qwen2.5-3B-Instruct", "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(TARGET)
target = AutoModelForCausalLM.from_pretrained(TARGET, dtype=torch.float16).to("cuda").eval()
draft  = AutoModelForCausalLM.from_pretrained(DRAFT,  dtype=torch.float16).to("cuda").eval()
print(f"loaded - GPU memory: {torch.cuda.memory_allocated()/1e9:.1f} GB (3B target + 0.5B draft)")

In [ ]:
PROMPTS = {
    "code":     "Write a Python function that parses an ISO-8601 date string and returns a datetime. Include a docstring.",
    "factual":  "List the planets of the solar system with one notable fact about each.",
    "creative": "Invent a myth about why cats purr, in the style of an ancient folk tale.",
}

def generate(prompt, assistant=None, n_new=200, **kw):
    # two-step tokenization: robust across transformers v4/v5
    text = tok.apply_chat_template([{"role":"user","content":prompt}],
                                   add_generation_prompt=True, tokenize=False)
    ids = tok(text, return_tensors="pt").input_ids.to("cuda")
    torch.cuda.synchronize(); t0 = time.perf_counter()
    with torch.no_grad():
        out = target.generate(ids, max_new_tokens=n_new, do_sample=False,
                              assistant_model=assistant, pad_token_id=tok.eos_token_id, **kw)
    torch.cuda.synchronize()
    n = out.shape[1] - ids.shape[1]
    return n / (time.perf_counter() - t0), tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True)

generate("warmup", n_new=16); generate("warmup", assistant=draft, n_new=16)

print(f"{'prompt type':<10}{'baseline':>10}{'speculative':>13}{'speedup':>9}")
print("-" * 42)
for name, p in PROMPTS.items():
    base_tps, base_txt = generate(p)
    spec_tps, spec_txt = generate(p, assistant=draft)
    same = "identical" if base_txt == spec_txt else "differs*"
    print(f"{name:<10}{base_tps:>8.1f}/s{spec_tps:>11.1f}/s{spec_tps/base_tps:>8.2f}x  ({same})")

print("\n(greedy decoding => outputs should be identical - the speedup is genuinely free;")
print(" *tiny fp16 nondeterminism can occasionally flip a token, same distribution either way)")

**What you should see:** code/factual prompts ~1.5–2.5× on a T4, creative prose less. The
*outputs match the baseline* — that's the lossless guarantee doing its job. The cost: the draft
model's 1 GB of VRAM, and wasted work whenever guesses miss.

## Part 3 · Prompt-lookup decoding — the draft model you already have

For summarization, extraction, document Q&A, and code *editing*, the output heavily copies spans of
the **input**. So skip the draft model entirely: when the last few generated tokens match an n-gram
in the prompt, propose the tokens that followed it there. Zero extra memory, zero extra model — and
it composes with the same verify step. This ships in `transformers` (`prompt_lookup_num_tokens`)
and vLLM (`method: "ngram"`):

In [ ]:
LONG_DOC = ("Quarterly report: The Atlantic Widget Company produced 48,200 widgets in Q3, "
            "up from 41,700 in Q2. Northern division output rose 9 percent to 21,300 units, "
            "while the Southern division delivered 18,400 units despite a two-week retooling "
            "pause. Export orders reached 11,050 units, led by demand from Rotterdam and Osaka. "
            "Defect rates fell to 0.8 percent following the new inspection line, and on-time "
            "delivery improved to 96.5 percent. Management expects Q4 output of 52,000 units, "
            "contingent on stable supply of aluminum fasteners from the Tacoma facility.")

task = f"{LONG_DOC}\n\nQuote the exact sentences about division output and about defect rates."

base_tps, _   = generate(task)
pl_tps, txt   = generate(task, prompt_lookup_num_tokens=10)
print(f"baseline:       {base_tps:5.1f} tok/s")
print(f"prompt-lookup:  {pl_tps:5.1f} tok/s   ({pl_tps/base_tps:.2f}x, zero extra models)")
print("\n", txt[:300])

Copy-heavy tasks routinely see **2–4×** from this one argument. It's the highest
ROI-per-line-of-code trick in this whole series.

## Part 4 · 2025 state of the art: don't draft with a *model* at all

The classic two-model setup has awkward costs: a separate draft to deploy, and it can only be
*sequentially* wrong. The field moved toward **self-speculation** — heads grafted onto the target
itself:

| Method | Idea | Notes |
|---|---|---|
| **Medusa** (2024) | bolt N extra LM heads onto the target; head *i* guesses token *t+i*; verify a small *tree* of candidates in one pass | simple, needs head fine-tuning |
| **EAGLE 1/2/3** (2024–25) | a 1-layer draft head that autoregresses on the target's *hidden features* (much more informative than tokens); EAGLE-2/3 grow **dynamic trees** guided by draft confidence | current OSS state of the art — ~3–6× reported; in vLLM & SGLang |
| **MTP** (DeepSeek V3/R1) | train the model *from the start* with multi-token-prediction heads, reuse them as the draft at inference | speculation as a pre-training decision — the direction frontier labs went |

Tree verification is the shared upgrade: instead of one k-token guess, verify a whole branching
bush of candidate continuations in a single pass (attention masks make cousins invisible to each
other), then keep the longest accepted path.

In **vLLM**, speculation is a config away (draft model, EAGLE, or n-gram):

```python
LLM(model="Qwen/Qwen2.5-3B-Instruct", dtype="half", max_model_len=2048,
    speculative_config={"method": "ngram", "num_speculative_tokens": 5,
                        "prompt_lookup_max": 4})  # n-gram: no extra model, great for RAG
# or: {"model": "Qwen/Qwen2.5-0.5B-Instruct", "num_speculative_tokens": 5}
# or: {"method": "eagle", "model": "<an EAGLE head trained for your target>", ...}
```

Watch `acceptance rate` in vLLM's logged spec-decode metrics: it's the α from Part 1, live. One
caveat before you turn it on in production: speculation trades *spare* compute for latency — it
shines at low/medium load, but at very high batch sizes there is no spare compute, and engines may
auto-disable it.

## Part 5 · The serving frontier (what "latest infra" means in 2025)

A map of the ideas the current generation of serving systems is built on — you've now measured the
foundations of every one of them:

- **Chunked prefill** (default in vLLM V1): slice long prefills into chunks scheduled *between*
  decode steps, so a 30k-token arrival doesn't freeze everyone's streaming. It's the fix for the
  prefill-vs-decode fight from notebook 21.
- **Prefill/decode disaggregation**: run compute-hungry prefill and bandwidth-hungry decode on
  *different GPU pools*, shipping the KV cache between them (Mooncake — which serves Kimi,
  DistServe, **NVIDIA Dynamo**, vLLM's P/D mode). Each pool gets hardware and batch shapes tuned
  for its phase; "goodput per GPU" replaces raw throughput as the metric.
- **KV cache as infrastructure** (LMCache, Mooncake's global cache, Dynamo's KV-aware routing):
  KV blocks get offloaded to CPU/NVMe, shared across nodes, and requests get *routed to the replica
  that already holds their prefix*. Your chat history is literally cached state in a distributed store.
- **RadixAttention** (SGLang): all live + recent KV organized in one radix tree, so *any* shared
  prefix across users/turns/branches is reused automatically — the generalization of notebook 22's
  prefix caching.
- **Structured output at kernel speed** (xgrammar, llguidance): constrained decoding (JSON schemas,
  grammars) compiled to token masks with near-zero overhead — agents and function-calling made cheap.
- **Attention variants for cheap KV**: GQA everywhere, **MLA** (DeepSeek) compressing KV ~10×,
  sliding-window + hybrid layers (Gemma/Llama 4-era) — architecture and serving co-designed, as in
  the notebook 21 calculator.

## Recap of the whole serving arc (notebooks 21→24)

| # | Bottleneck | Fix you ran |
|---|---|---|
| 21 | decode is bandwidth-bound; KV memory is scarce | measured it: cache on/off, TTFT/TPOT, batching, CB simulator |
| 22 | KV fragmentation + straggler batches | vLLM: PagedAttention, continuous batching, prefix cache, real server |
| 23 | weight bytes dominate decode reads | AWQ/GPTQ int4: 4× memory, faster decode, quiz-checked |
| 24 | idle compute during decode | speculative + prompt-lookup decoding, losslessly faster |

That's the modern serving stack: **page it, batch it, shrink it, speculate it.**

### Further reading
- [Fast Inference via Speculative Decoding](https://arxiv.org/abs/2211.17192) (Leviathan et al.) · [Chen et al.](https://arxiv.org/abs/2302.01318) — the twin original papers, proofs included
- [EAGLE-3](https://arxiv.org/abs/2503.01840) · [Medusa](https://arxiv.org/abs/2401.10774) · [Prompt lookup decoding](https://github.com/apoorvumang/prompt-lookup-decoding)
- [DeepSeek-V3 report](https://arxiv.org/abs/2412.19437) (§ MTP) · [Mooncake](https://arxiv.org/abs/2407.00079) · [DistServe](https://arxiv.org/abs/2401.09670)
- [vLLM speculative decoding docs](https://docs.vllm.ai/en/latest/features/spec_decode.html) · [SGLang](https://github.com/sgl-project/sglang)

🏁 **You've reached the end of the serving arc — and the notebook series.** [Back to the learning path](README.md).